In [1]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

import cv2
import numpy as np
import time
from pynput.mouse import Button, Controller

2024-10-01 00:52:01.109419: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-01 00:52:01.109494: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-01 00:52:01.137151: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-01 00:52:01.194369: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-01 00:52:02.226356: W tensorflow/compiler/tf2

In [2]:
import shutil
import os

# Define the path to the folder you want to delete
folder_path = '/mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/augmented'

# Check if the folder exists
if os.path.exists(folder_path):
    # Delete the folder and all its contents
    shutil.rmtree(folder_path)
    print(f"Folder '{folder_path}' has been deleted.")
else:
    print(f"Folder '{folder_path}' does not exist.")

Folder '/mnt/Main Drive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/augmented' does not exist.


In [3]:
base_options = python.BaseOptions(model_asset_path='/mnt/Main Drive/Codes/Deep Learning/Gesture_control/Mediapipe_Gesture/Model_w_Brightness_200/gesture_recognizer.task')
options = vision.GestureRecognizerOptions(base_options=base_options)
recognizer = vision.GestureRecognizer.create_from_options(options)

I0000 00:00:1727724124.138350  113503 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1727724124.140855  113715 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1), renderer: AMD Radeon Graphics (renoir, LLVM 15.0.7, DRM 3.54, 6.5.0-44-generic)
W0000 00:00:1727724124.147694  113503 gesture_recognizer_graph.cc:129] Hand Gesture Recognizer contains CPU only ops. Sets HandGestureRecognizerGraph acceleration to Xnnpack.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


**Functions**

In [4]:
def Get_Gesture(frame):

    image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
    recognition_result = recognizer.recognize(image)

    if recognition_result.gestures:
        top_gesture = recognition_result.gestures[0][0].category_name
        # print(top_gesture)
    else:
        top_gesture = "Nothing"
        
    return top_gesture

In [5]:
def Image_Processing(frame,hands,top_gesture):

    image = cv2.cvtColor(cv2.flip(frame,1),cv2.COLOR_BGR2RGB)
    image.flags.writeable = False

    results = hands.process(image)
    
    image.flags.writeable = True
    image = cv2.cvtColor(image,cv2.COLOR_RGB2BGR)

    fontScale = 2
    fontFace = cv2.FONT_HERSHEY_PLAIN
    fontColor = (0,255,0)
    fontThickness = 2

    cv2.putText(image,top_gesture,(0,30),fontFace,fontScale,fontColor,fontThickness,cv2.LINE_AA)


    return image,results

In [6]:
class Gesture_Action:

    def __init__(self):
        self.mouse = Controller()
        self.w = 640
        self.h = 480
        self.x = 0  # Initialize previous x coordinate
        self.y = 0  # Initialize previous y coordinate


    def Move_Cursor(self, results):
        if results.multi_hand_landmarks:
            hand_landmarks = results.multi_hand_landmarks[0]  # Access the first hand
            current_x = hand_landmarks.landmark[8].x * self.w  # Scale current x coordinate
            current_y = hand_landmarks.landmark[8].y * self.h  # Scale current y coordinate
            cx, cy = current_x - self.x, current_y - self.y  # Calculate difference
            self.mouse.move(cx * 5, cy *5)  # Move cursor by the difference
            self.x, self.y = current_x, current_y  # Update previous coordinates


    def Gesture_Action(self,top_gesture,results):
        mouse = self.mouse
        
        if top_gesture == "Index":
            self.Move_Cursor(results)
        elif top_gesture == "Pinch":
            mouse.press(Button.left)
        elif top_gesture == "Close_L_shape":
            pass
        elif top_gesture == "L_shape":
            pass
        elif top_gesture == "Close_Palm":
            pass
        elif top_gesture == "Open_Palm":
            pass
        elif top_gesture == "scissors":
            pass

**Mouse**

In [7]:
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands
s = "http://127.0.0.1:4747/video"
s1 = 0
cap = cv2.VideoCapture(s1)

G = Gesture_Action()


In [8]:
with mp_hands.Hands(static_image_mode=False, max_num_hands=4, min_detection_confidence=0.7, min_tracking_confidence=0.5) as hands:
    while cap.isOpened():
        ret,frame = cap.read()
        
        if not ret:
            print("Ignoring empty camera frame.")
            break

        top_gesture = Get_Gesture(frame)

        image,results = Image_Processing(frame,hands,top_gesture)
        
        h,w,c = image.shape

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:

                # cx, cy = hand_landmarks.landmark[8].x, hand_landmarks.landmark[8].y

                hand_landmarks,mp_drawing.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS)
               
            G.Gesture_Action(top_gesture,results)
            cv2.imshow('Hand Tracking', image)

        else:
            cv2.imshow('Hand Tracking', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1727724124.385846  113503 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1727724124.387020  113751 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1), renderer: AMD Radeon Graphics (renoir, LLVM 15.0.7, DRM 3.54, 6.5.0-44-generic)
/home/neutrino/miniconda3/envs/Ml/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Prototype Code


In [9]:
# with mp_hands.Hands(static_image_mode=False, max_num_hands=2, min_detection_confidence=0.7, min_tracking_confidence=0.5) as hands:
#     while cap.isOpened():
#         ret,frame = cap.read()
        
#         if not ret:
#             print("Ignoring empty camera frame.")
#             break

#         # image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
#         # recognition_result = recognizer.recognize(image)
#         # if recognition_result.gestures:
#         #     top_gesture = recognition_result.gestures[0][0].category_name
#         #     # print(top_gesture)

#         # else:
#         #     top_gesture = "Nothing"
#         top_gesture = Get_Gesture(frame)


#         # image = cv2.cvtColor(cv2.flip(frame,1),cv2.COLOR_BGR2RGB)

#         # image.flags.writeable = False

#         # results = hands.process(image)
#         # image.flags.writeable = True

#         # image = cv2.cvtColor(image,cv2.COLOR_RGB2BGR)
#         image,results = Image_Processing(frame,hands,top_gesture)
#         # fontScale = 2
#         # fontFace = cv2.FONT_HERSHEY_PLAIN
#         # fontColor = (0,255,0)
#         # fontThickness = 2

#         # Draw bounding box
#         h,w,c = image.shape


#         if results.multi_hand_landmarks:
#             for hand_landmarks in results.multi_hand_landmarks:
#                 cx, cy = int(hand_landmarks.landmark[8].x * w), int(hand_landmarks.landmark[8].y * h)

#                 hand_landmarks,mp_drawing.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS)
               
        
 
    

#             # check = GEST_V(image,hand_landmarks.landmark[8],hand_landmarks.landmark[5],hand_landmarks.landmark[12],hand_landmarks.landmark[9])
#             # print (check)
#             # if check == 1:
#             screen_cx, screen_cy = calibrate_coordinates_sensitivity(cx, cy)
#             mouse.position = (screen_cx, screen_cy)

#             cv2.imshow('Hand Tracking', image)
#             # pyautogui.moveTo(screen_cx, screen_cy)

#         else:
#             cv2.imshow('Hand Tracking', image)

#         if cv2.waitKey(10) & 0xFF == ord('q'):
#             break

# cap.release()
# cv2.destroyAllWindows()